In [3]:
from collections import defaultdict

# Sample search query corpus
corpus = [
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit near me",
    "best places to visit during summer",
    "best restaurants in chennai",
    "best hotels in india",
    "places to visit in kerala"
]

N = 5
D = 0.75

# Sparse storage
ngram_count = defaultdict(int)
context_count = defaultdict(int)
followers = defaultdict(set)
continuation = defaultdict(set)
vocab = set()

for sentence in corpus:

    words = ["<s>"] * (N - 1) + sentence.lower().split() + ["</s>"]

    for word in words:
        vocab.add(word)

    for i in range(len(words) - N + 1):

        ngram = tuple(words[i:i + N])
        context = tuple(words[i:i + N - 1])

        ngram_count[ngram] += 1
        context_count[context] += 1

        followers[context].add(ngram[-1])
        continuation[ngram[-1]].add(context)
        
def kneser_ney(context, word):

    if len(context) == 0:
        if word in vocab:
            return 1 / len(vocab)
        else:
            return 1 / (len(vocab) + 1)

    ngram = tuple(context) + (word,)

    count = ngram_count.get(ngram, 0)
    history = context_count.get(tuple(context), 0)

    if history == 0:
        return kneser_ney(context[1:], word)

    first = max(count - D, 0) / history

    lamb = (D * len(followers[tuple(context)])) / history

    backoff = kneser_ney(context[1:], word)

    return first + lamb * backoff

def predict(query, top=5):

    words = ["<s>"] * (N - 1) + query.lower().split()

    context = tuple(words[-(N - 1):])

    scores = []

    for word in vocab:

        if word == "<s>":
            continue

        p = kneser_ney(context, word)
        scores.append((word, p))

    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:top]

query = "best places to visit"

print("Query :", query)
print("\nSuggestions:\n")

for word, prob in predict(query):
    print(word, ":", round(prob, 4))

Query : best places to visit

Suggestions:

in : 0.3477
during : 0.0977
near : 0.0977
to : 0.0352
visit : 0.0352
